# Swin-T + UPerNet (ADE20K) — DIMER semantic-segmentation tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/swin-segmentation-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/swin-segmentation-pipeline/blob/main/tutorials/swin_segmentation_task_inference.ipynb) [![Python 3.10 required](https://img.shields.io/badge/Python-3.10%20required-3776ab?style=flat&logo=python&logoColor=white)](https://github.com/kurtvalcorza/swin-segmentation-pipeline/blob/main/README.md) [![Checkpoint](https://img.shields.io/badge/OpenMMLab-upernet__swin--t__ade20k__512x512__160k-ffcc4d?style=flat)](https://github.com/open-mmlab/mmsegmentation/tree/v1.2.2/configs/swin) [![Upstream](https://img.shields.io/badge/Upstream-microsoft%2FSwin--Transformer-181717?style=flat&logo=github&logoColor=white)](https://github.com/microsoft/Swin-Transformer) [![arXiv](https://img.shields.io/badge/arXiv-2103.14030-b31b1b.svg)](https://arxiv.org/abs/2103.14030)

**Profile:** `TASK-INFERENCE`  
**Notebook specification:** DIMER Notebook Specification 1.1 — **standalone** (§3.6)  
**Capability:** pretrained ADE20K-150 semantic segmentation (one class index per pixel) using the pinned OpenMMLab `swin-tiny-patch4-window7-in1k-pre_upernet_8xb2-160k_ade20k-512x512` checkpoint through the repository's `DimerSwinSegmenter` API

**This notebook is standalone.** It carries the repository's package (2 modules under `src/dimer_swin_segmentation/`, at revision `ae0bba29a6ea`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the OpenMMLab checkpoint host (`download.openmmlab.com`) at the immutable MMSegmentation release-tag commit `c685fe6767c4cadf6b051983ca6208f1b9d1ccb8` (~240 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

At inference the pinned Swin-T backbone + UPerNet decode head maps one RGB image to one ADE20K class index (`0..149`) per pixel; the public API returns that 2-D `uint8` mask and the classes present, and exposes no per-pixel confidence. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream OpenMMLab checkpoint supplies the weights and the pinned MMSegmentation 1.2.2 package supplies the config and test pipeline, and the carried package adds snapshot verification, runtime-version checks, input validation, a fixed output contract and the `semantic_iou`, `majority_class_baseline`, `validate_inputs` and `evaluation_report` helpers.

**Trust boundary (MOD12).** The checkpoint is a code-capable PyTorch `.pth` serialization. The carried `verify_snapshot` re-hashes it against the inline manifest (and against the digest the package has always pinned in `MODEL_SPEC`) before the pinned MMSegmentation loader deserializes it inside `mmengine`; the deserialization call is upstream's, not the package's, and is **not** a `weights_only` load. A matching digest proves byte identity with the pinned OpenMMLab distribution, not publisher authenticity — run this notebook only where that pinned source is trusted.

The default sample is a synthetic scene generated in code (no download, no ground truth), so its mask is demonstration (plumbing) evidence, not a correctness or benchmark claim. A gated option fetches two labelled ADE20K validation fixtures from the Hugging Face Hub at an immutable dataset commit instead and evaluates mean IoU on them.

**Learning objectives:** install the pinned Python 3.10 OpenMMLab CPU runtime, read what the carried package guarantees, resolve and digest-verify the immutable OpenMMLab checkpoint, generate a synthetic default input (or opt into the labelled ADE20K fixtures / a BYOD image) and validate it into an input manifest, run segmentation through the public API, read a class-index mask and its class coverage correctly, produce an evaluation report that is `sample-sanity` with `semantic_iou` only when ground-truth masks exist and `not-measurable` otherwise, and export class-index masks plus machine-readable results and provenance.

**This notebook does not demonstrate:** instance or panoptic segmentation, object detection, depth, open-vocabulary segmentation, image classification, or any training or fine-tuning. The label space is fixed to the 150 ADE20K categories; pixels of other things still receive an ADE20K label, and the API exposes no per-pixel uncertainty.

## Prerequisites

- **Runtime:** a **CPython 3.10** Jupyter kernel on Linux (the notebook asserts `sys.version_info[:2] == (3, 10)` and stops otherwise). The qualified OpenMMLab stack — torch 2.1.2 (CPU build), MMCV 2.1.0, MMEngine 0.10.7, MMSegmentation 1.2.2, NumPy 1.26.4 — has prebuilt wheels for Python 3.10 only; `pip` cannot change the interpreter, so a Python 3.11+ kernel (including current default Colab runtimes) is unsupported and the pinned install fails there. CPU is the default and only qualified path; no GPU is required. The pinned torch/mmcv wheels are the largest downloads of the run.
- **Knowledge:** basic Python and image handling; what a per-pixel class map and an intersection-over-union metric are.
- **Data:** the default sample is a deterministic 512×384 synthetic scene (gradient background plus flat-coloured shapes) generated in code, so nothing is downloaded and there is no ground truth. Two optional gates are off by default so the sample path runs top-to-bottom without interaction: `USE_ADE20K_FIXTURES` fetches two public ADE20K validation fixtures (`ADE_val_00000001`, `ADE_val_00000002`: two JPEG images and their PNG annotation maps, ~94 KB in total) from the Hugging Face dataset `hf-internal-testing/fixtures_ade20k` at the immutable commit `850d349e…`, verifies each file's SHA-256, and enables mean-IoU evaluation; `USE_BYOD` uploads one image file decodable by Pillow (PNG/JPEG/WebP and similar, at most 64 megapixels). Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** the OpenMMLab checkpoint host (`download.openmmlab.com`) only, to fetch the pinned `open-mmlab/mmsegmentation:swin-tiny-patch4-window7-in1k-pre_upernet_8xb2-160k_ade20k-512x512` snapshot (~240 MB in total) at revision `c685fe6767c4…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's tools/pins.txt at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `numpy` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    '--extra-index-url',
    'https://download.pytorch.org/whl/cpu',
    '--find-links',
    'https://download.openmmlab.com/mmcv/dist/cpu/torch2.1/index.html',
    'torch==2.1.2+cpu',
    'torchvision==0.16.2+cpu',
    'mmengine==0.10.7',
    'mmcv==2.1.0',
    'mmsegmentation==1.2.2',
    'ftfy==6.3.1',
    'regex==2024.11.6',
    'numpy==1.26.4',
    'opencv-python==4.10.0.84',
    'pillow==11.3.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'swin-segmentation-pipeline',
    'repository_revision': 'ae0bba29a6eae0078c2569d7efd980e92c2d9cfd',
    'embedded_module': 'src/dimer_swin_segmentation/runtime.py',
    'embedded_modules': ['src/dimer_swin_segmentation/metrics.py', 'src/dimer_swin_segmentation/runtime.py'],
    'module_sha256': '0560d607fbb8939ab0f1ed8ae326a5b8ec56e661a36699244ad0f4c8c392d767',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '1.1',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, numpy
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'numpy': numpy.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/dimer_swin_segmentation/` @ `ae0bba29a6ea`)

The next 2 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/2:** `src/dimer_swin_segmentation/metrics.py`

In [ ]:
"""Semantic-segmentation metric helpers carried by the standalone tutorial (NOTEBOOK_SPEC 1.1 EVAL2).

`semantic_iou` is the repository's segmentation metric: per-class intersection/union aggregated over
every (prediction, reference) pair, mean IoU over the classes that occur in the sample (union > 0),
and pixel accuracy over the labelled pixels. `majority_class_baseline` scores the constant predictor
that paints every pixel with the sample's most frequent reference class — a descriptive reference
derived from the same tiny sample, not an independent benchmark. `ade20k_raw_to_indices` maps a raw
ADE20K annotation (`0` = unlabelled, `1..150` = classes) onto the model's class indices `0..149` with
`0` sent to the ignore index, the same `reduce_zero_label=True` convention the pinned MMSegmentation
config uses. Pure NumPy; no model logic lives here.
"""

from __future__ import annotations

from collections.abc import Sequence
from typing import Any

import numpy as np

ADE20K_NUM_CLASSES = 150
IGNORE_INDEX = 255


def ade20k_raw_to_indices(raw: np.ndarray, *, num_classes: int = ADE20K_NUM_CLASSES) -> np.ndarray:
    """Raw ADE20K labels (0 = ignore, 1..num_classes) -> model indices 0..num_classes-1, ignore -> 255."""
    array = np.asarray(raw)
    if array.ndim != 2:
        raise ValueError(f"expected a 2-D label map, got shape {array.shape}")
    if array.min() < 0 or array.max() > num_classes:
        observed = f"{int(array.min())}..{int(array.max())}"
        raise ValueError(f"raw ADE20K labels must lie in 0..{num_classes}: {observed}")
    out = np.full(array.shape, IGNORE_INDEX, dtype=np.uint16)
    labelled = array > 0
    out[labelled] = array[labelled].astype(np.uint16) - 1
    return out


def _pairs(
    predictions: Sequence[np.ndarray], references: Sequence[np.ndarray], ignore_index: int
) -> list[tuple[np.ndarray, np.ndarray, np.ndarray]]:
    if len(predictions) != len(references):
        raise ValueError(f"{len(predictions)} predictions but {len(references)} references")
    if not predictions:
        raise ValueError("at least one prediction/reference pair is required")
    out = []
    for index, (prediction, reference) in enumerate(zip(predictions, references, strict=True)):
        pred = np.asarray(prediction)
        ref = np.asarray(reference)
        if pred.ndim != 2 or pred.shape != ref.shape:
            raise ValueError(
                f"pair {index}: prediction {pred.shape} and reference {ref.shape} must be equal 2-D shapes"
            )
        out.append((pred.astype(np.int64), ref.astype(np.int64), ref != ignore_index))
    return out


def semantic_iou(
    predictions: Sequence[np.ndarray],
    references: Sequence[np.ndarray],
    *,
    num_classes: int = ADE20K_NUM_CLASSES,
    ignore_index: int = IGNORE_INDEX,
) -> dict[str, Any]:
    """Aggregate per-class IoU, mean IoU over classes present (union > 0) and pixel accuracy.

    `predictions` and `references` are equal-length sequences of equal-shape 2-D class-index maps
    (model indices `0..num_classes-1`); reference pixels equal to `ignore_index` are excluded.
    """
    intersections = np.zeros(num_classes, dtype=np.int64)
    unions = np.zeros(num_classes, dtype=np.int64)
    correct = 0
    valid_pixels = 0
    for pred, ref, valid in _pairs(predictions, references, ignore_index):
        if valid.any() and (pred[valid].min() < 0 or pred[valid].max() >= num_classes):
            raise ValueError(f"prediction indices must lie in 0..{num_classes - 1}")
        if valid.any() and ref[valid].max() >= num_classes:
            raise ValueError(f"reference indices must lie in 0..{num_classes - 1} or equal ignore_index")
        correct += int(((pred == ref) & valid).sum())
        valid_pixels += int(valid.sum())
        pred_hist = np.bincount(pred[valid], minlength=num_classes)[:num_classes]
        ref_hist = np.bincount(ref[valid], minlength=num_classes)[:num_classes]
        both = np.bincount(pred[valid & (pred == ref)], minlength=num_classes)[:num_classes]
        intersections += both
        unions += pred_hist + ref_hist - both
    present = np.flatnonzero(unions > 0)
    per_class = [
        {
            "class_id": int(class_id),
            "intersection_pixels": int(intersections[class_id]),
            "union_pixels": int(unions[class_id]),
            "iou": float(intersections[class_id] / unions[class_id]),
        }
        for class_id in present
    ]
    return {
        "miou": float(np.mean([row["iou"] for row in per_class])) if per_class else float("nan"),
        "pixel_accuracy": float(correct / valid_pixels) if valid_pixels else float("nan"),
        "valid_pixels": valid_pixels,
        "classes_with_union": len(per_class),
        "n_images": len(predictions),
        "per_class": per_class,
    }


def majority_class_baseline(
    references: Sequence[np.ndarray],
    *,
    num_classes: int = ADE20K_NUM_CLASSES,
    ignore_index: int = IGNORE_INDEX,
) -> dict[str, Any]:
    """Score the constant predictor painting every pixel with the sample's most frequent reference class."""
    refs = [np.asarray(reference).astype(np.int64) for reference in references]
    if not refs:
        raise ValueError("at least one reference is required")
    counts = np.zeros(num_classes, dtype=np.int64)
    for ref in refs:
        valid = ref != ignore_index
        counts += np.bincount(ref[valid], minlength=num_classes)[:num_classes]
    if not counts.any():
        raise ValueError("references contain no labelled pixels")
    majority = int(counts.argmax())
    constant = [np.full(ref.shape, majority, dtype=np.int64) for ref in refs]
    scored = semantic_iou(constant, refs, num_classes=num_classes, ignore_index=ignore_index)
    return {"majority_class_id": majority, "miou": scored["miou"], "pixel_accuracy": scored["pixel_accuracy"]}

**Module 2/2:** `src/dimer_swin_segmentation/runtime.py` (carried verbatim; see the note above)

In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import os
import tempfile
import urllib.request
from collections.abc import Callable, Iterable
from dataclasses import dataclass
from pathlib import Path

import numpy as np
from PIL import Image

# standalone rewrite (build_notebook.py): `from .metrics import majority_class_baseline, semantic_iou` removed — names are kernel globals defined by the carried modules

MODEL_SPEC = {
    "runtime_id": "swin-t-upernet-ade20k-mmseg-v1.2.2",
    "architecture": "Swin-T + UPerNet",
    "task": "semantic-segmentation",
    "dataset": "ADE20K",
    "mmsegmentation_version": "1.2.2",
    "mmcv_version": "2.1.0",
    "mmengine_version": "0.10.7",
    "torch_version": "2.1.2",
    "config": "swin/swin-tiny-patch4-window7-in1k-pre_upernet_8xb2-160k_ade20k-512x512.py",
    "config_source_revision": "open-mmlab/mmsegmentation@v1.2.2",
    "checkpoint_url": "https://download.openmmlab.com/mmsegmentation/v0.5/swin/upernet_swin_tiny_patch4_window7_512x512_160k_ade20k_pretrain_224x224_1K/upernet_swin_tiny_patch4_window7_512x512_160k_ade20k_pretrain_224x224_1K_20210531_112542-e380ad3e.pth",
    "checkpoint_size_bytes": 240154742,
    "checkpoint_sha256": "e380ad3e5d94060d89e4b62b5d393cdcc7f1f3406b1d46bcab547d3c276b6064",
    "upstream_reported_miou": 44.41,
    "num_classes": 150,
}


@dataclass(frozen=True)
class SegmentationResult:
    image_id: str
    mask: np.ndarray
    classes_present: tuple[int, ...]

    def summary(self) -> dict:
        return {
            "image_id": self.image_id,
            "shape": list(self.mask.shape),
            "classes_present": list(self.classes_present),
            "num_classes_present": len(self.classes_present),
        }

    def save_mask(self, path: str | os.PathLike) -> Path:
        target = Path(path)
        target.parent.mkdir(parents=True, exist_ok=True)
        Image.fromarray(self.mask.astype(np.uint8), mode="L").save(target)
        return target


def _sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def _version(dist: str) -> str:
    return importlib.metadata.version(dist)


# ---------------------------------------------------------------------------------------------
# Fleet snapshot scheme (DIMER standalone carrier). The identity constants below name the OpenMMLab
# distribution: MODEL_ID is the config recipe inside the pinned package, MODEL_REVISION the upstream
# git commit of that package's release tag (the config source), and the manifest pins the checkpoint
# bytes. The checkpoint host is download.openmmlab.com, not the Hugging Face Hub, so the staging
# downloader is the pinned URL in MODEL_SPEC rather than hf_hub_download.
# ---------------------------------------------------------------------------------------------
MODEL_ID = "open-mmlab/mmsegmentation:swin-tiny-patch4-window7-in1k-pre_upernet_8xb2-160k_ade20k-512x512"
MODEL_REVISION = "c685fe6767c4cadf6b051983ca6208f1b9d1ccb8"
MODEL_LICENSE = "Apache-2.0"
MODEL_KEY = "swin-t-upernet-ade20k"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHTS_FILE = "upernet_swin_tiny_patch4_window7_512x512_160k_ade20k_pretrain_224x224_1K_20210531_112542-e380ad3e.pth"
MAX_PIXELS = 64_000_000  # validate_image ceiling


def verify_snapshot(path: str | os.PathLike | None = None) -> dict:
    """Check a local snapshot against its manifest; raise naming the first mismatch.

    The checkpoint entry must also carry the digest MODEL_SPEC has always pinned, so the manifest
    cannot silently re-point the runtime at different bytes.
    """
    root = Path(path or DEFAULT_WEIGHTS_DIR)
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    entries = {entry["path"]: entry for entry in manifest.get("files", [])}
    pinned = entries.get(WEIGHTS_FILE)
    if pinned is None or pinned["sha256"] != MODEL_SPEC["checkpoint_sha256"] or pinned["bytes"] != MODEL_SPEC["checkpoint_size_bytes"]:
        raise ValueError(f"manifest entry for {WEIGHTS_FILE} does not match the checkpoint digest pinned in MODEL_SPEC")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _openmmlab_download(relative_path: str, root: Path) -> None:
    """Fetch the pinned OpenMMLab checkpoint into the snapshot directory (the only manifest entry)."""
    if relative_path != WEIGHTS_FILE:
        raise ValueError(f"no pinned download source for {relative_path}")
    target = root / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary_name = tempfile.mkstemp(prefix="dimer-swin-", suffix=".pth", dir=root)
    os.close(fd)
    temporary = Path(temporary_name)
    try:
        with urllib.request.urlopen(MODEL_SPEC["checkpoint_url"], timeout=120) as response, temporary.open("wb") as out:
            while True:
                block = response.read(1024 * 1024)
                if not block:
                    break
                out.write(block)
        temporary.replace(target)
    finally:
        if temporary.exists():
            temporary.unlink()


def stage_missing_files(
    path: str | os.PathLike | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the checkpoint). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them from {MODEL_SPEC['checkpoint_url']}"
        )
    fetch = downloader or _openmmlab_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def verify_runtime_versions() -> dict[str, str]:
    expected = {
        "mmsegmentation": MODEL_SPEC["mmsegmentation_version"],
        "mmcv": MODEL_SPEC["mmcv_version"],
        "mmengine": MODEL_SPEC["mmengine_version"],
    }
    actual = {name: _version(name) for name in expected}
    for name, expected_version in expected.items():
        if actual[name] != expected_version:
            raise RuntimeError(
                f"Unsupported {name} {actual[name]}; this runtime is qualified for {expected_version}."
            )
    return actual


def resolve_packaged_config() -> Path:
    import mmseg

    config = (
        Path(mmseg.__file__).resolve().parent
        / ".mim"
        / "configs"
        / MODEL_SPEC["config"]
    )
    if not config.is_file():
        raise RuntimeError(
            f"The pinned MMSegmentation package does not contain the expected config: {config}"
        )
    return config


def acquire_verified_checkpoint(cache_dir: str | os.PathLike = ".dimer-models") -> Path:
    root = Path(cache_dir).expanduser().resolve()
    root.mkdir(parents=True, exist_ok=True)
    target = root / Path(MODEL_SPEC["checkpoint_url"]).name

    def valid(path: Path) -> bool:
        return (
            path.is_file()
            and path.stat().st_size == MODEL_SPEC["checkpoint_size_bytes"]
            and _sha256(path) == MODEL_SPEC["checkpoint_sha256"]
        )

    if valid(target):
        return target
    if target.exists():
        target.unlink()

    fd, temporary_name = tempfile.mkstemp(prefix="dimer-swin-segmentation-", suffix=".pth", dir=root)
    os.close(fd)
    temporary = Path(temporary_name)
    try:
        with urllib.request.urlopen(MODEL_SPEC["checkpoint_url"], timeout=120) as response, temporary.open("wb") as out:
            while True:
                block = response.read(1024 * 1024)
                if not block:
                    break
                out.write(block)
        if temporary.stat().st_size != MODEL_SPEC["checkpoint_size_bytes"]:
            raise RuntimeError(
                f"Checkpoint size mismatch: got {temporary.stat().st_size}, expected {MODEL_SPEC['checkpoint_size_bytes']}."
            )
        digest = _sha256(temporary)
        if digest != MODEL_SPEC["checkpoint_sha256"]:
            raise RuntimeError(
                f"Checkpoint SHA-256 mismatch: got {digest}, expected {MODEL_SPEC['checkpoint_sha256']}."
            )
        temporary.replace(target)
    finally:
        if temporary.exists():
            temporary.unlink()
    return target


def validate_image(path: str | os.PathLike, *, max_pixels: int = 64_000_000) -> dict:
    image_path = Path(path).expanduser().resolve()
    if not image_path.is_file():
        raise ValueError(f"Image does not exist: {image_path}")
    try:
        with Image.open(image_path) as image:
            image.verify()
        with Image.open(image_path) as image:
            width, height = image.size
            mode = image.mode
    except Exception as exc:
        raise ValueError(f"Input is not a readable image: {image_path}: {exc}") from exc
    if width < 1 or height < 1:
        raise ValueError(f"Image dimensions must be positive; got {width}x{height}.")
    if width * height > max_pixels:
        raise ValueError(
            f"Image has {width * height:,} pixels; ceiling is {max_pixels:,}. Resize before inference."
        )
    return {"path": str(image_path), "width": width, "height": height, "mode": mode}


INPUT_SCHEMA: dict = {
    "input": "one or more image files readable by Pillow (any mode), given by path",
    "pixels": [1, MAX_PIXELS],
    "classes": "150 ADE20K categories in MMSegmentation order",
    "preprocessing": "MMSegmentation test pipeline of the pinned config (resize to 512-scale, normalise); nothing is altered by this module",
}


def validate_inputs(
    images: str | os.PathLike | Iterable[str | os.PathLike],
    *,
    names: Iterable[str] | None = None,
) -> dict:
    """Validation stage: return the input manifest (schema, per-image observations, verdict).

    Rejection is reported by raising exactly as ``predict`` would (``validate_image`` for each image);
    a caller that wants the finding recorded catches the exception and stores ``str(exc)`` under
    ``findings``.
    """
    paths = [images] if isinstance(images, (str, os.PathLike)) else list(images)
    observed = [validate_image(p) for p in paths]
    ids = list(names) if names is not None else [Path(o["path"]).name for o in observed]
    if len(ids) != len(observed):
        raise ValueError("names must have one entry per image")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [{"id": ids[i], **o} for i, o in enumerate(observed)],
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    results: Iterable[SegmentationResult | dict],
    ground_truth: Iterable[np.ndarray] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``ground_truth`` (one 2-D map of model class indices ``0..149`` per result, in the same order,
    ``255`` = ignore — see ``metrics.ade20k_raw_to_indices``) the report carries ``semantic_iou`` (mean
    IoU over the classes present, pixel accuracy, per-class IoU) and the ``majority_class_baseline`` with
    the verdict ``sample-sanity``; ``ground_truth`` then requires ``SegmentationResult`` inputs (the
    masks). Without it the verdict is ``not-measurable`` (EVAL9) and the report says what labelled data
    would make the task measurable.
    """
    items = list(results)
    rows = [r.summary() if isinstance(r, SegmentationResult) else dict(r) for r in items]
    base = {
        "task": "ADE20K-150 semantic segmentation",
        "score_semantics": "argmax class per pixel; no per-pixel confidence is exposed",
        "sample_kind": sample_kind,
        "n_images": len(rows),
        "context": {
            "upstream_reported_miou": MODEL_SPEC["upstream_reported_miou"],
            "note": "upstream full-ADE20K mIoU as reported by OpenMMLab; not measured here",
        },
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if ground_truth is None:
        return {
            **base,
            "metrics": [],
            "baselines": [],
            "verdict": "not-measurable",
            "reason": "no ground-truth masks were supplied for the evaluated images",
            "needs": (
                "ADE20K-indexed ground-truth masks (model indices 0..149, 255 = ignore) for the evaluated images, "
                "scored with semantic_iou (mean IoU over classes present, pixel accuracy, per-class IoU) against the "
                "majority_class_baseline; a labelled set from the deployment domain for any generalisable claim"
            ),
        }
    if not all(isinstance(item, SegmentationResult) for item in items):
        raise TypeError("ground_truth evaluation needs SegmentationResult inputs (the predicted masks)")
    references = [np.asarray(reference) for reference in ground_truth]
    scored = semantic_iou([item.mask for item in items], references, num_classes=MODEL_SPEC["num_classes"])
    baseline = majority_class_baseline(references, num_classes=MODEL_SPEC["num_classes"])
    estimation = (
        f"single labelled sample of {scored['n_images']} image(s) / {scored['valid_pixels']} labelled pixels, "
        "aggregate IoU, no dispersion estimate"
    )
    return {
        **base,
        "metrics": [
            {"id": "semantic_iou", "metric": "miou", "value": scored["miou"], "estimation": estimation},
            {"id": "semantic_iou", "metric": "pixel_accuracy", "value": scored["pixel_accuracy"], "estimation": estimation},
        ],
        "per_class": scored["per_class"],
        "classes_with_union": scored["classes_with_union"],
        "baselines": [
            {
                "id": "majority_class_baseline",
                "majority_class_id": baseline["majority_class_id"],
                "miou": baseline["miou"],
                "pixel_accuracy": baseline["pixel_accuracy"],
                "note": "constant predictor of the sample's most frequent class; derived from the same sample",
            }
        ],
        "verdict": "sample-sanity",
        "reason": f"{scored['n_images']} labelled image(s) from the tutorial sample; not a benchmark",
        "needs": "a representative labelled holdout from the deployment domain for any generalisable mIoU claim",
    }


class DimerSwinSegmenter:
    """Public semantic-segmentation task-inference API for the pinned Swin-T UPerNet model.

    The upstream `.pth` checkpoint is code-capable PyTorch serialization. This
    runtime verifies exact size and SHA-256 before MMSegmentation deserializes
    it. Digest verification establishes byte identity, not publisher authenticity.
    """

    def __init__(
        self,
        *,
        cache_dir: str | os.PathLike = ".dimer-models",
        device: str = "cpu",
        checkpoint: str | os.PathLike | None = None,
        source: str = "openmmlab-cache",
    ):
        versions = verify_runtime_versions()
        checkpoint = Path(checkpoint) if checkpoint is not None else acquire_verified_checkpoint(cache_dir)
        config = resolve_packaged_config()
        from mmseg.apis import init_model

        self.model = init_model(str(config), str(checkpoint), device=device)
        self.device = device
        self.checkpoint = checkpoint
        self.config = config
        self.versions = versions
        self.source = source
        self.classes = tuple(self.model.dataset_meta.get("classes", ()))
        if len(self.classes) != MODEL_SPEC["num_classes"]:
            raise RuntimeError(
                f"Expected {MODEL_SPEC['num_classes']} ADE20K classes, got {len(self.classes)}."
            )

    @classmethod
    def from_pretrained(
        cls,
        *,
        device: str = "cpu",
        weights_dir: str | os.PathLike | None = None,
        allow_download: bool = False,
    ) -> DimerSwinSegmenter:
        """Load from the fleet snapshot directory: stage absent manifest entries (only with
        ``allow_download=True``, from the pinned OpenMMLab URL), re-hash every entry against the
        manifest and MODEL_SPEC, then deserialise the checkpoint through the pinned OpenMMLab loader.
        The ``.pth`` is code-capable PyTorch serialization: digest verification fixes the bytes, not
        the author — see the class docstring.
        """
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        if not (root / MANIFEST_NAME).is_file():
            raise FileNotFoundError(
                f"no snapshot manifest at {root}; use DimerSwinSegmenter(cache_dir=...) for the cache path"
            )
        stage_missing_files(root, allow_download=allow_download)
        verify_snapshot(root)
        return cls(device=device, checkpoint=root / WEIGHTS_FILE, source="local-snapshot")

    def predict(self, image: str | os.PathLike) -> SegmentationResult:
        info = validate_image(image)
        from mmseg.apis import inference_model

        sample = inference_model(self.model, info["path"])
        mask = sample.pred_sem_seg.data.squeeze(0).detach().cpu().numpy().astype(np.uint8, copy=False)
        if mask.ndim != 2:
            raise RuntimeError(f"Expected a 2-D semantic mask; got shape {mask.shape}.")
        classes_present = tuple(int(x) for x in np.unique(mask))
        return SegmentationResult(
            image_id=Path(info["path"]).name,
            mask=mask,
            classes_present=classes_present,
        )

    def provenance(self) -> dict:
        import platform

        import torch

        return {
            "runtime": MODEL_SPEC,
            "effective": {
                "python": platform.python_version(),
                "torch": torch.__version__,
                **self.versions,
                "device": self.device,
                "source": self.source,
                "classes": list(self.classes),
                "checkpoint_path": str(self.checkpoint),
                "checkpoint_sha256": _sha256(self.checkpoint),
            },
            "output_semantics": "Per-pixel ADE20K class index in [0, 149]. No calibrated per-pixel uncertainty is exported by this runtime.",
        }

    def write_provenance(self, path: str | os.PathLike) -> Path:
        target = Path(path)
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(json.dumps(self.provenance(), indent=2) + "\n")
        return target

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `1`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the OpenMMLab checkpoint host (`download.openmmlab.com`) **at MMSegmentation release-tag commit `c685fe6767c4…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `DimerSwinSegmenter.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "swin-t-upernet-ade20k",
  "modelId": "open-mmlab/mmsegmentation:swin-tiny-patch4-window7-in1k-pre_upernet_8xb2-160k_ade20k-512x512",
  "revision": "c685fe6767c4cadf6b051983ca6208f1b9d1ccb8",
  "files": [
    {
      "path": "upernet_swin_tiny_patch4_window7_512x512_160k_ade20k_pretrain_224x224_1K_20210531_112542-e380ad3e.pth",
      "bytes": 240154742,
      "sha256": "e380ad3e5d94060d89e4b62b5d393cdcc7f1f3406b1d46bcab547d3c276b6064"
    }
  ],
  "totalBytes": 240154742
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = DimerSwinSegmenter.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Confirm the qualified runtime

The carried package fails closed on version drift: `verify_runtime_versions` (called when the model was constructed above) compares the installed `mmseg`, `mmcv` and `mmengine` distributions with the versions pinned in `MODEL_SPEC`, and this cell additionally asserts the Python 3.10 interpreter the OpenMMLab wheels were built for. Look for a dictionary reporting Python 3.10.x, `torch` 2.1.2+cpu, MMSegmentation 1.2.2, MMCV 2.1.0, MMEngine 0.10.7, 150 classes, the verified checkpoint file name and the `local-snapshot` source.

In [ ]:
import sys

if sys.version_info[:2] != (3, 10):
    raise RuntimeError(f'Python 3.10 is required by the qualified OpenMMLab runtime (see Prerequisites); this kernel is {sys.version.split()[0]}. Use a Python 3.10 kernel.')
print({'python': platform.python_version(), 'torch': torch.__version__, 'numpy': numpy.__version__, **pipe.versions, 'classes': len(pipe.classes), 'checkpoint': pipe.checkpoint.name, 'source': pipe.source, 'device': pipe.device})

## 5. Generate the synthetic sample, or opt into the ADE20K fixtures / BYOD

The default sample is **synthetic**: a deterministic 512×384 scene (a red→green gradient background with three flat-coloured shapes) drawn in code and written to `sample/`, so it needs no download and its SHA-256 is printed for the record. It depicts no ADE20K scene, so it has **no ground truth**: whatever mask the model returns is a sanity check that the input contract, preprocessing and forward pass work, not a correctness measurement. Two gates are off by default. `USE_ADE20K_FIXTURES` fetches two real labelled ADE20K validation examples from the Hugging Face Hub at an immutable dataset commit, refuses any file whose SHA-256 differs from the recorded digest, checks that each annotation map has the image's shape and raw labels in `0..150`, and converts the raw labels with the carried `ade20k_raw_to_indices` (`0` → ignore, `1..150` → `0..149`, the pinned config's `reduce_zero_label=True` convention) — no split is invented or changed. `USE_BYOD` uploads one image; BYOD has no ground truth unless you supply a matching annotation map yourself. Look for a dictionary naming the sample kind, the image files, their digests and whether ground truth exists.

In [ ]:
import hashlib
from pathlib import Path

from PIL import Image, ImageDraw

USE_BYOD = False  # @param {type:"boolean"}
USE_ADE20K_FIXTURES = False  # @param {type:"boolean"}
FIXTURES_COMMIT = '850d349e5038f291284e7999fcacbedc0922534b'
FIXTURES_BASE = f'https://huggingface.co/datasets/hf-internal-testing/fixtures_ade20k/resolve/{FIXTURES_COMMIT}'
FIXTURES_SHA256 = {
    'ADE_val_00000001.jpg': '99f7af15a7bd66f3d2ad8b98f1b41941c27407a35d303f9b70724cb231a36098',
    'ADE_val_00000001.png': '7724c8b985ba9978e968fed231d74fb9e72abd4179f3c4b58bb87c525efb9ae7',
    'ADE_val_00000002.jpg': 'ce373a18513e5a357d7fd2d2f70e1db2249417635068e108cd85ddca08195f30',
    'ADE_val_00000002.png': 'db1497cd6acd98c61178fbff5611da0604877f55cb7b72655d4e4ecac386904f',
}
sample_dir = Path('sample')
sample_dir.mkdir(exist_ok=True)
ground_truth = None
if USE_BYOD and USE_ADE20K_FIXTURES:
    raise ValueError('Enable at most one of USE_BYOD and USE_ADE20K_FIXTURES.')
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    upload_name = next(iter(uploaded))
    image_path = sample_dir / Path(upload_name).name
    image_path.write_bytes(uploaded[upload_name])
    image_paths = [image_path]
    sample_kind = 'BYOD'
elif USE_ADE20K_FIXTURES:
    import urllib.request

    for name, expected in FIXTURES_SHA256.items():
        target = sample_dir / name
        if not target.exists():
            urllib.request.urlretrieve(f'{FIXTURES_BASE}/{name}', target)
        observed = hashlib.sha256(target.read_bytes()).hexdigest()
        if observed != expected:
            raise RuntimeError(f'{name}: digest {observed} != pinned {expected}; refusing the fixture')
    image_paths = [sample_dir / 'ADE_val_00000001.jpg', sample_dir / 'ADE_val_00000002.jpg']
    ground_truth = []
    for path in image_paths:
        with Image.open(path) as opened:
            width, height = opened.size
        with Image.open(path.with_suffix('.png')) as annotation:
            raw = numpy.array(annotation)
        if raw.shape != (height, width):
            raise RuntimeError(f'{path.name}: annotation shape {raw.shape} != image shape {(height, width)}')
        ground_truth.append(ade20k_raw_to_indices(raw))
    sample_kind = 'ADE20K-fixtures'
else:
    # Deterministic synthetic scene: no randomness, so no seed is needed and the digest is stable.
    width, height = 512, 384
    ramp = numpy.linspace(0.0, 255.0, width)
    red = numpy.tile(ramp, (height, 1))
    green = numpy.tile(numpy.linspace(0.0, 255.0, height)[:, None], (1, width))
    blue = (red + green) / 2.0
    array = numpy.rint(numpy.stack([red, green, blue], axis=-1)).astype(numpy.uint8)
    scene = Image.fromarray(array, mode='RGB')
    draw = ImageDraw.Draw(scene)
    draw.rectangle([40, 240, 220, 360], fill=(20, 20, 20))
    draw.ellipse([300, 60, 460, 220], fill=(240, 240, 240))
    draw.polygon([(260, 370), (330, 250), (400, 370)], fill=(30, 90, 200))
    image_path = sample_dir / 'synthetic_scene_512x384.png'
    scene.save(image_path)
    image_paths = [image_path]
    sample_kind = 'synthetic'
image_names = [path.name for path in image_paths]
sample_sha256 = {path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in image_paths}
print({'sample_kind': sample_kind, 'images': image_names, 'sha256': sample_sha256, 'ground_truth': None if ground_truth is None else [list(mask.shape) for mask in ground_truth]})

## 6. Validate the input → input manifest

`validate_inputs` is the package's public validation stage: it applies exactly the checks `predict` applies — per image, `validate_image` (the file exists, Pillow can decode it, positive dimensions, at most `MAX_PIXELS` = 64,000,000 pixels) — and returns an **input manifest** naming the schema and ceilings, each input's observed path, size and mode, and the verdict. The manifest is written to `outputs/swin_segmentation_task_inference_input_manifest.json`. To show what rejection looks like, the cell also validates a path that does not exist and records the package's own error message as a finding. Inside the package every accepted image goes through the pinned config's MMSegmentation test pipeline (resize to the 512 scale, normalise); nothing is dropped or altered by the package itself, and the returned mask has the input image's height and width.

In [ ]:
import json
import os

os.makedirs('outputs', exist_ok=True)
print({'ceilings': {'MAX_PIXELS': MAX_PIXELS, 'pixels': INPUT_SCHEMA['pixels'], 'classes': len(pipe.classes)}})
input_manifest = validate_inputs(image_paths, names=image_names)
# Demonstrate rejection on an input that breaks the contract; the finding is recorded, not swallowed.
try:
    validate_inputs(sample_dir / 'does-not-exist.png')
except ValueError as exc:
    input_manifest['findings'].append({'input': 'missing-file-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/swin_segmentation_task_inference_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print(json.dumps(input_manifest, indent=2))

## 7. Segment

`predict` runs one image through the pinned MMSegmentation inference path and returns a `SegmentationResult`: a 2-D `uint8` `mask` of ADE20K class indices (`0..149`, the per-pixel argmax of the decode head) with the image's height and width, plus the sorted `classes_present`. The mask is a hard decision — the API exposes **no per-pixel confidence or calibrated uncertainty**, and the package ships no threshold. Inference is deterministic given the same weights, device and library versions (`model.eval()`, no sampling); CPU kernel choices can flip near-tied pixels. Each mask is saved as a PNG class-index raster (`outputs/swin_segmentation_task_inference_<image>_semantic.png`, value = class index). Look for the per-image shape, the number of classes present and the top classes by pixel coverage; on the synthetic scene expect a handful of large flat regions with arbitrary labels.

In [ ]:
results = [pipe.predict(path) for path in image_paths]
coverage = []
for path, result in zip(image_paths, results, strict=True):
    mask_path = result.save_mask(f'outputs/swin_segmentation_task_inference_{path.stem}_semantic.png')
    values, counts = numpy.unique(result.mask, return_counts=True)
    order = numpy.argsort(counts)[::-1]
    for class_id, pixels in zip(values[order].tolist(), counts[order].tolist(), strict=True):
        coverage.append({'image_id': result.image_id, 'class_id': int(class_id), 'class_name': pipe.classes[int(class_id)], 'pixels': int(pixels), 'fraction': float(pixels / result.mask.size)})
    print({**result.summary(), 'mask_file': mask_path.name})
for row in coverage[:8]:
    print(f"{row['image_id']:<24} class {row['class_id']:>3} {row['class_name']:<20} {row['fraction']:.3f} of pixels")

## 8. Evaluate → evaluation report

`evaluation_report` is the package's public evaluation stage and always produces a report. When ground-truth masks exist (the `USE_ADE20K_FIXTURES` path) it carries `semantic_iou` — the repository's metric helper: mean IoU over the classes present in the sample (union > 0), pixel accuracy over the labelled pixels, and per-class IoU aggregated across the images — with the verdict `sample-sanity` and the `majority_class_baseline` (a constant predictor of the sample's most frequent class, derived from the same two images, so a descriptive reference rather than an independent benchmark): a two-image tutorial metric with high sampling variance, not comparable to the upstream full-ADE20K mIoU of 44.41 that `MODEL_SPEC` records as upstream-reported context. On the synthetic default sample (and on BYOD without an annotation map) no metric exists, so the verdict is `not-measurable` and the report states what would make the task measurable. The report is written to `outputs/swin_segmentation_task_inference_evaluation_report.json`.

In [ ]:
report = evaluation_report(results, ground_truth, sample_kind=sample_kind)
with open('outputs/swin_segmentation_task_inference_evaluation_report.json', 'w', encoding='utf-8') as handle:
    json.dump(report, handle, indent=2, ensure_ascii=False)
print(json.dumps({key: value for key, value in report.items() if key != 'per_class'}, indent=2))
if report['verdict'] == 'not-measurable':
    print('No ground-truth masks were supplied, so semantic_iou is not computed; the masks above are sanity evidence only.')
else:
    for row in sorted(report['per_class'], key=lambda row: row['union_pixels'], reverse=True)[:10]:
        print(f"class {row['class_id']:>3} {pipe.classes[row['class_id']]:<20} IoU {row['iou']:.3f}  union {row['union_pixels']} px")

## 9. Export outputs and provenance

Machine-readable JSON preserves each mask's summary and file name, the per-image class coverage, the evaluation report (including per-class IoU when measured), the input manifest, the sample identity and digests, the notebook's source (repository, revision, embedded module digest, generator), the model identifier, the immutable revision label, the checkpoint digest the package verified, and the runtime identity (Python, `torch`, `mmseg`, `mmcv`, `mmengine`, device). The class coverage is also written as CSV with explicit `image_id`, `class_id`, `class_name`, `pixels` and `fraction` columns, ordered by coverage within each image, so the mask semantics survive downstream use. No credentials are recorded.

In [ ]:
import csv

payload = {
    'predictions': [{**result.summary(), 'mask_file': f'swin_segmentation_task_inference_{Path(result.image_id).stem}_semantic.png'} for result in results],
    'class_coverage': coverage,
    'evaluation_report': report,
    'input_manifest': input_manifest,
    'sample': {'kind': sample_kind, 'images': image_names, 'sha256': sample_sha256, 'ground_truth': ground_truth is not None},
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'checkpoint_sha256': MODEL_SPEC['checkpoint_sha256'],
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'numpy': numpy.__version__,
        **pipe.versions,
        'device': pipe.device,
    },
}
with open('outputs/swin_segmentation_task_inference_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/swin_segmentation_task_inference_class_coverage.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['image_id', 'class_id', 'class_name', 'pixels', 'fraction'])
    for row in coverage:
        writer.writerow([row['image_id'], row['class_id'], row['class_name'], row['pixels'], f"{row['fraction']:.6f}"])
print(sorted(os.listdir('outputs')))

## Interpretation and limits

Each pixel receives one class from the fixed 150-category ADE20K label space by per-pixel argmax; the API exposes **no per-pixel confidence** and the package ships no threshold. On the synthetic scene the mask is meaningless by construction and the evaluation report says `not-measurable`; a `semantic_iou` value from the two ADE20K fixtures is tutorial evidence for those images and must not be generalized to a domain, camera, resolution or scene composition. Scenes outside ADE20K's indoor/outdoor photographic distribution, thin structures, class boundaries and domain shifts (medical, aerial, line art) all degrade results in ways the package does not detect. The package provides no instance, panoptic, detection, depth, classification, or training capability, and the `.pth` checkpoint remains a code-capable serialization whose digest check fixes the bytes, not the author.

Successful execution proves that the recorded repository revision's package, carried in this notebook, can acquire and digest-verify the pinned OpenMMLab checkpoint, assert the qualified Python 3.10 / MMSegmentation 1.2.2 runtime, validate the demonstrated input, execute the public segmentation path, and emit the shown machine-readable outputs and class-index masks in the tested runtime — without the repository being reachable. It does **not** establish benchmark superiority, reproduction of the upstream ADE20K result, calibrated per-pixel uncertainty, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** enable `USE_ADE20K_FIXTURES` to see the report switch to `sample-sanity` with `semantic_iou` (mean IoU, pixel accuracy, per-class IoU) against the `majority_class_baseline`; enable `USE_BYOD` with a photograph from your own domain and inspect which classes dominate the coverage table before investing in annotation; annotate a small holdout from that domain with ADE20K indices and compare its mean IoU with the fixture value.

## References

- Repository README: https://github.com/kurtvalcorza/swin-segmentation-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/swin-segmentation-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/swin-segmentation-pipeline/blob/main/docs/WEIGHTS.md
- Pinned checkpoint (OpenMMLab host): https://download.openmmlab.com/mmsegmentation/v0.5/swin/upernet_swin_tiny_patch4_window7_512x512_160k_ade20k_pretrain_224x224_1K/upernet_swin_tiny_patch4_window7_512x512_160k_ade20k_pretrain_224x224_1K_20210531_112542-e380ad3e.pth
- Config source (MMSegmentation v1.2.2, `configs/swin`): https://github.com/open-mmlab/mmsegmentation/tree/v1.2.2/configs/swin
- ADE20K fixtures (Hugging Face dataset, immutable commit): https://huggingface.co/datasets/hf-internal-testing/fixtures_ade20k/tree/850d349e5038f291284e7999fcacbedc0922534b
- Upstream project: https://github.com/microsoft/Swin-Transformer
- Swin Transformer: Hierarchical Vision Transformer using Shifted Windows: https://arxiv.org/abs/2103.14030
- Unified Perceptual Parsing for Scene Understanding (UPerNet): https://arxiv.org/abs/1807.10221
- Scene Parsing through ADE20K Dataset: https://arxiv.org/abs/1608.05442